# Data Mining - Lab 5
## Өгөгдлийн чанар: дутуу ба алдаатай өгөгдөл илрүүлэх

**Dataset:** students.csv  

Хичээлийн слайд дээрх `students.csv` жишээг өргөтгөсөн 315 мөртэй dataset. Өгөгдөлд дутуу утга, буруу утга, давхардал, тэмдэгтийн зөрүү, огнооны янз бүрийн формат зориудаар оруулсан.

### Lab зорилго
- Өгөгдлийн чанарын 4 шалгуурыг (бүрэн байдал, үнэн зөв байдал, нийцтэй байдал, давхардал) практикт шалгах
- Нууц (hidden) дутуу утгыг (`N/A`, `?`, хоосон зай) илрүүлэх
- Алдаатай утга, хэтрэлт (outlier), давхардлыг олох
- Илрүүлсэн асуудлыг засах, эсвэл хасах шийдвэр гаргах
- Цэвэрлэгээг data leakage-гүйгээр хийх

## 0) Dataset унших ба ерөнхий шалгалт

Эхлээд файлыг **ямар ч хөрвүүлэлтгүйгээр** (бүх утгыг текстээр) уншиж, өгөгдөл яг ямар байгааг харна.

Баганууд: `id`, `age`, `score`, `attendance`, `city`, `program`, `enroll_date`

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("students.csv", dtype=str, keep_default_na=False)
raw.head(10)

    id   age score attendance         city       program enroll_date
0  101    19    82         85           УБ  Программчлал  2025/09/08
1  102          75         67  Улаанбаатар        Сүлжээ  2025/09/10
2  103    -5   110         86           УБ        Сүлжээ  2025/09/25
3  103    -5   110         86           УБ        Сүлжээ  2025/09/25
4  104    20   N/A         80       Дархан        Сүлжээ  2025/09/30
5  105  хорь    91         80          УБ   Программчлал  2025/09/07
6  106    20    67         89  Улаанбаатар        Дизайн  2025/09/02
7  107    22    90        100       Дархан  Программчлал  2025/09/18
8  108    22    66         89           УБ        Дизайн  2025-09-29
9  109    22    72         97  Улаанбаатар        Дизайн  2025/09/22

## 1) Өгөгдлийн чанарын эхний шалгалт

Өгөгдлийн чанарыг дараах шалгуураар хэмжинэ:

- **Бүрэн байдал (completeness):** дутуу утга байна уу?
- **Үнэн зөв байдал (accuracy):** утга зөв, боломжит мужид байна уу?
- **Нийцтэй байдал (consistency):** нэг зүйл нэг хэлбэрээр бичигдсэн үү?
- **Давхардал (uniqueness):** нэг бичлэг хоёр удаа орсон уу?

### TASK 1: Хэмжээ ба төрөл

1. Dataset хэдэн мөр, хэдэн багана вэ?
2. Дараах `df`-ийг `na_values` ашиглан дахин уншина. Энд `?` ба `N/A`-г дутуу утга гэж үзнэ.
3. `df.dtypes`-ийг харна. `age` ба `score` багана яагаад тоон төрөл биш байна вэ? (үгээр)

**Хариу:** `age` баганад `хорь`, `twenty`, `score` баганад `67%` зэрэг тоонд хөрвөхгүй текст утга байгаа тул pandas баганыг текст (object) гэж таньдаг.

In [2]:
print(raw.shape)
df = pd.read_csv("students.csv", na_values=["?", "N/A"])
print(df.dtypes)

(315, 7)
id               int64
age                str
score              str
attendance     float64
city               str
program            str
enroll_date        str
dtype: object


## 2) Дутуу утга илрүүлэх (Missing values)

`isna()` нь зөвхөн жинхэнэ `NaN`-г олдог. Харин хоосон зай (`" "`) зэрэг нууц дутуу утга үлдэж болно.

### TASK 2: Багана бүрийн дутуу утга

1. Багана бүрт хэдэн дутуу утга байгааг ол.
2. Хувиар (%) харуул, `missing_pct` нэртэй хүснэгт үүсгэ.

In [3]:
miss_cnt = df.isna().sum()
missing_pct = (miss_cnt / len(df) * 100).round(1)
pd.DataFrame({"missing": miss_cnt, "missing_pct": missing_pct})

             missing  missing_pct
id                 0          0.0
age               26          8.3
score             26          8.3
attendance        13          4.1
city               4          1.3
program            0          0.0
enroll_date        4          1.3

### TASK 3: Нууц дутуу утга (hidden missing)

`city` баганад зөвхөн хоосон зайгаас бүрдсэн утга байна. Эдгээрийг `NaN` болго.

Зөвлөмж: regex `^\s*$` нь "зөвхөн хоосон зайгаас бүрдсэн" утгыг илтгэнэ.

1. Солихын өмнө `city` багана дахь `NaN`-ийн тоог хэвлэ.
2. Бүх баганад хоосон зайг `NaN` болго.
3. Дараа нь дахин тоолж, хэд нэмэгдсэнийг тайлбарла.

In [4]:
before = df["city"].isna().sum()
df = df.replace(r"^\s*$", np.nan, regex=True)
after = df["city"].isna().sum()
print("өмнө:", before, "дараа:", after)

өмнө: 4 дараа: 9


## 3) Дутуу утга яагаад үүссэн бэ?

Дутуу утгыг устгахаасаа өмнө **механизмыг** ойлгох хэрэгтэй:

- **MCAR:** дутуу байх нь санамсаргүй, бусад утгаас хамаарахгүй
- **MAR:** дутуу байх нь өөр ажиглагдсан баганаас хамаарна
- **MNAR:** дутуу байх нь тухайн утгын өөрөөс нь хамаарна

### TASK 4: Дутуу score хаана төвлөрч байна вэ?

1. `score` баганыг тоон болгож (`to_numeric`) `score_num` үүсгэ.
2. `program` бүрээр `score_num`-ийн дутуу утгын **хувийг** ол.
3. Аль хөтөлбөр дээр илүү их дутуу байна вэ? Энэ нь MCAR уу, MAR уу? (үгээр)

**Хариу:** Дизайн хөтөлбөрт дутуу оноо хамгийн олон (19.3%), бусад хөтөлбөрт ойролцоогоор 5%. Дутуу байдал хөтөлбөрөөс хамаарч байгаа тул MAR гэж үзнэ. MNAR-ийг өгөгдлөөс бүрэн үгүйсгэх боломжгүй.

In [5]:
df["score_num"] = pd.to_numeric(df["score"], errors="coerce")
miss_by_prog = df.groupby("program")["score_num"].apply(lambda s: s.isna().mean() * 100).round(1)
miss_by_prog

program
Дизайн          19.3
Программчлал     4.8
Сүлжээ           5.8
Name: score_num, dtype: float64

## 4) Алдаатай утга илрүүлэх (Erroneous data)

Алдаа гурван хэлбэрээр гарна:

1. **Буруу төрөл:** тоо байх ёстой баганад текст (`хорь`, `twenty`, `85%`)
2. **Мужаас гарсан утга:** нас сөрөг, оноо 100-аас их
3. **Нийцгүй утга:** нэг ойлголт олон янзаар бичигдсэн

### TASK 5: Буруу төрөл

1. `age` баганыг `to_numeric(errors="coerce")`-ээр хөрвүүл. Үр дүнг `age_num` гэж нэрлэ.
2. Хөрвүүлэхэд **шинээр** `NaN` болсон (өмнө нь дутуу биш байсан) утгуудыг гарга.
3. Тэдгээр нь ямар утгууд вэ? Яагаад автоматаар засаж болохгүй вэ? (үгээр)

**Хариу:** `хорь` ба `twenty` нь утгатай ч үгийг тоонд хөрвүүлэх найдвартай дүрэм байхгүй. `67%` нь хувь тул хөрвүүлж болох ч энэ ажилд дутуу гэж тэмдэглэв. Автоматаар засахад эрсдэлтэй.

In [6]:
df["age_num"] = pd.to_numeric(df["age"], errors="coerce")
bad_type = df[df["age"].notna() & df["age_num"].isna()]
bad_type["age"].value_counts()

age
хорь      4
twenty    2
Name: count, dtype: int64

### TASK 6: Мужийн шалгалт (range check)

Дүрэм:

- Нас: **16 ба 40-ийн хооронд** (хоёр захыг оруулна)
- Оноо: **0 ба 100-ийн хооронд**
- Ирц (`attendance`): **0 ба 100-ийн хооронд**

1. Дүрэм бүрийг зөрчсөн мөрийн тоог ол (дутуу утгыг тооцохгүй).
2. Оноо 100-аас их мөрүүдийг харуул.
3. Оноо яг `0` байгаа мөрүүд алдаа уу, дутуу утга уу? Шийдвэрээ үгээр тайлбарла.

**Хариу:** Оноо 0 нь 0–100 мужид багтаж байгаа тул мужийн алдаа биш. Гэхдээ орлуулсан утга байж магадгүй тул шалгах хэрэгтэй. Одоогоор хадгалсан.

In [7]:
df["attendance_num"] = pd.to_numeric(df["attendance"], errors="coerce")
bad_age = df["age_num"].notna() & ~df["age_num"].between(16, 40)
bad_score = df["score_num"].notna() & ~df["score_num"].between(0, 100)
bad_att = df["attendance_num"].notna() & ~df["attendance_num"].between(0, 100)
print("age:", bad_age.sum(), "score:", bad_score.sum(), "attendance:", bad_att.sum())
df.loc[df["score_num"] > 100, ["id", "score", "program"]]

age: 12 score: 10 attendance: 2


      id score       program
2    103   110        Сүлжээ
3    103   110        Сүлжээ
17   117   110  Программчлал
108  202   900  Программчлал
113  207   900  Программчлал
198  289   150  Программчлал
239  327   110        Дизайн
251  339   150        Сүлжээ
275  361   110  Программчлал

### TASK 7: Тэмдэгтийн нийцгүй байдал (city)

`city` баганад Улаанбаатар олон янзаар бичигдсэн (`УБ`, ` УБ `, `улаанбаатар`, ...).

1. `value_counts(dropna=False)`-ээр бүх утгыг хар.
2. `str.strip()` ба `str.lower()`-ээр цэвэрлэ.
3. `УБ`-г `улаанбаатар`-тай нэг болгохын тулд `replace` / `map` ашигла.
4. Цэвэрлэсний дараа хэдэн өвөрмөц хот үлдэв?

In [8]:
print(df["city"].value_counts(dropna=False))
df["city_clean"] = df["city"].str.strip().str.lower()
df["city_clean"] = df["city_clean"].replace({"уб": "улаанбаатар"})
print(df["city_clean"].value_counts(dropna=False))
print("өвөрмөц хот:", df["city_clean"].nunique())

city
Улаанбаатар    123
Дархан          59
Эрдэнэт         38
УБ              33
Ховд            31
 УБ              9
NaN              9
улаанбаатар      6
Дархан           4
ДАРХАН           3
Name: count, dtype: int64
city_clean
улаанбаатар    171
дархан          66
эрдэнэт         38
ховд            31
NaN              9
Name: count, dtype: int64
өвөрмөц хот: 4


## 5) Огнооны формат

`enroll_date` баганад гурван хэлбэр холилдсон: `2025/09/12`, `12/09/2025`, `2025-09-12`, мөн бичгийн алдаатай утга байна.

Нэг `to_datetime` дуудлагаар бүгдийг уншиж болохгүй. Формат бүрийг тусад нь оролдоно.

### TASK 8: Огноог стандартчилах

1. Формат бүрээр `to_datetime(format=..., errors="coerce")` оролд.
2. Нэг формат амжилтгүй бол дараагийнхыг ашиглахын тулд `fillna` ашигла.
3. Бүх формат амжилтгүй болсон мөрийн тоог ол. Эдгээр нь юу вэ?

In [9]:
d = df["enroll_date"]
d1 = pd.to_datetime(d, format="%Y/%m/%d", errors="coerce")
d2 = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
d3 = pd.to_datetime(d, format="%d/%m/%Y", errors="coerce")
df["enroll_dt"] = d1.fillna(d2).fillna(d3)
failed = df[df["enroll_date"].notna() & df["enroll_dt"].isna()]
print("амжилтгүй:", len(failed))
failed["enroll_date"].value_counts()

амжилтгүй: 2


enroll_date
20250/09/25    1
20250/09/12    1
Name: count, dtype: int64

## 6) Давхардал илрүүлэх (Duplicates)

Давхардал хоёр төрөлтэй:

- **Бүрэн давхардал:** бүх багана ижил
- **Түлхүүрийн давхардал:** `id` ижил боловч бусад утга зөрүүтэй (зөрчил)

### TASK 9: Давхардал

1. Бүрэн давхардсан **илүү** мөрийн тоог ол (`duplicated()`, анхных нь хадгална).
2. Давхардсан бүх мөрийг (анхных нь ч гэсэн) харахын тулд `keep=False` ашигла.
3. `id` ижил боловч мөр нь өөр байгаа тохиолдлыг ол. Зөрчилтэй `id` хэд вэ?
4. Бүрэн давхардлыг арилгасны дараа хэдэн мөр үлдэв?

In [10]:
cols = ["id", "age", "score", "attendance", "city", "program", "enroll_date"]
n_dup = df.duplicated(subset=cols).sum()
print("илүү давхардал:", n_dup)
df[df.duplicated(subset=cols, keep=False)].sort_values("id").head(6)

илүү давхардал: 12


     id age score  attendance         city       program enroll_date  score_num  age_num  attendance_num   city_clean  enroll_dt
2   103  -5   110        86.0           УБ        Сүлжээ  2025/09/25      110.0     -5.0            86.0  улаанбаатар 2025-09-25
3   103  -5   110        86.0           УБ        Сүлжээ  2025/09/25      110.0     -5.0            86.0  улаанбаатар 2025-09-25
31  131  22    74        83.0  Улаанбаатар        Дизайн  2025/09/15       74.0     22.0            83.0  улаанбаатар 2025-09-15
32  131  22    74        83.0  Улаанбаатар        Дизайн  2025/09/15       74.0     22.0            83.0  улаанбаатар 2025-09-15
64  163  24    69        78.0         Ховд  Программчлал  2025/09/06       69.0     24.0            78.0         ховд 2025-09-06
65  163  24    69        78.0         Ховд  Программчлал  2025/09/06       69.0     24.0            78.0         ховд 2025-09-06

In [11]:
dedup = df.drop_duplicates(subset=cols)
id_conflict = dedup[dedup.duplicated(subset="id", keep=False)].sort_values("id")
print("үлдсэн мөр:", len(dedup))
print("зөрчилтэй id:", id_conflict["id"].nunique())
id_conflict[["id", "score", "program"]]

үлдсэн мөр: 303
зөрчилтэй id: 3


      id score       program
86   183    65        Сүлжээ
87   183    74        Сүлжээ
101  197    63  Программчлал
102  197    72  Программчлал
176  268    65  Программчлал
177  268    74  Программчлал

## 7) Хэтрэлт (Outlier) илрүүлэх

Хэтрэлт нь алдаа байж болно, мөн жинхэнэ ховор утга байж болно. Тиймээс илрүүлээд **шууд устгахгүй**, шалтгааныг нь судална.

**IQR арга:**

- Q1 ба Q3 нь 25% ба 75%-ийн квантил, IQR = Q3 - Q1
- Доод хязгаар = Q1 - 1.5 * IQR
- Дээд хязгаар = Q3 + 1.5 * IQR
- Хязгаараас гарсан утга нь хэтрэлт

Жишээ: `[18, 19, 20, 21, 22, 22, 23, 60]`. Үүнд Q1 = 19.75, Q3 = 22.25, IQR = 2.5, хязгаар 16 ба 26 тул `60` хэтрэлт болно.

### TASK 10: IQR функц

1. Дээрх жишээн дээр функцээ шалга.
2. `dedup` дахь `age_num` дээр IQR хэтрэлтийг ол.
3. Гарсан утгуудаас аль нь мужийн алдаа (TASK 6), аль нь боломжит боловч ховор утга (жишээ нь 45, 52, 60) вэ?
4. Яагаад мужийн шалгалт ба IQR хоёр өөр зорилготой вэ? (үгээр)

**Хариу:** Мужийн алдаа нь -5, -1, 150, 190. Боломжтой ховор утга нь 45, 52, 60. Мужийн шалгалт нь мэдлэгт (домэйн) дүрэмд суурилдаг, IQR нь өгөгдлийн тархалтад суурилдаг тул хоёр арга өөр асуултад хариулна.

In [12]:
def iqr_bounds(s, k=1.5):
    s = s.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

demo = pd.Series([18, 19, 20, 21, 22, 22, 23, 60])
lo, hi = iqr_bounds(demo)
print("demo:", lo, hi, demo[(demo < lo) | (demo > hi)].tolist())

ages = dedup["age_num"]
lo, hi = iqr_bounds(ages)
print("age bounds:", lo, hi)
ages[(ages < lo) | (ages > hi)].sort_values().tolist()

demo: 16.0 26.0 [60]
age bounds: 16.0 24.0


[-5.0, -5.0, -5.0, -5.0, -1.0, 45.0, 52.0, 60.0, 150.0, 190.0]

### TASK 11: Z-score

Z-score = (x - дундаж) / стандарт хазайлт. `|z| > 3` бол хэтрэлт гэж үзнэ.

1. `score_num` дээр z-score тооц (зөвхөн 0 ба 100-ийн хоорондох утгууд дээр).
2. `|z| > 2.5` байгаа мөрийг ол.
3. IQR ба z-score хоёр ижил мөрүүдийг олсон уу? (үгээр)

**Хариу:** Бүрэн давхцахгүй. z-score нь 0, 7, 12 утгыг, IQR нь нэмээд 51-ийг илрүүлсэн.

In [13]:
valid = dedup["score_num"][dedup["score_num"].between(0, 100)]
z = (valid - valid.mean()) / valid.std()
valid[z.abs() > 2.5].sort_values().tolist()

[0.0, 0.0, 0.0, 7.0, 12.0]

## 8) Илрүүлсэн асуудлыг засах

Илрүүлсний дараа гурван сонголт бий:

- **Устгах:** мөр эсвэл баганыг хасна
- **Нөхөх (imputation):** дундаж, медиан, бүлгийн медиан, регресс
- **Тэмдэглэх:** `is_missing` гэсэн туг үүсгэнэ

Нөхөхдөө **хөтөлбөр бүрээр** нөхөх нь ерөнхий дундажаас илүү зөв байж болно, учир нь Дизайн хөтөлбөрийн оноо ялгаатай.

### TASK 12: Нөхөх арга

Эхлээд мужаас гарсан оноог `NaN` болго, дараа нь нөх.

1. `score_clean` үүсгэ. 0 ба 100-ийн гадна утгыг `NaN` болго (`where` эсвэл `mask`).
2. Нөхөхийн өмнө дутуу утгын тоог хэвлэ.
3. **Нийт медианаар** нөхсөн `score_global` үүсгэ.
4. **Хөтөлбөрийн медианаар** нөхсөн `score_group` үүсгэ (`groupby(...).transform("median")`).
5. Хоёр аргын Дизайн хөтөлбөрийн дунджийг харьцуул. Аль нь илүү бодитой вэ? (үгээр)

**Хариу:** Хөтөлбөрийн медианаар нөхсөн дундаж (68.3) нь дутуугүй дундаж (68.1)-д ойр. Нийт медианаар нөхсөн дундаж (69.2) зөрүү их тул хөтөлбөрийн медиан илүү бодитой.

In [14]:
work = dedup.copy()
work["score_clean"] = work["score_num"].where(work["score_num"].between(0, 100))
print("нөхөхийн өмнө NaN:", work["score_clean"].isna().sum())
work["score_global"] = work["score_clean"].fillna(work["score_clean"].median())
grp_med = work.groupby("program")["score_clean"].transform("median")
work["score_group"] = work["score_clean"].fillna(grp_med)
work.groupby("program")[["score_clean", "score_global", "score_group"]].mean().round(1)

нөхөхийн өмнө NaN: 37


              score_clean  score_global  score_group
program                                             
Дизайн               68.1          69.2         68.3
Программчлал         75.8          75.5         75.8
Сүлжээ               70.9          71.1         71.1

### TASK 13: Регрессээр нөхөх

`score` нь `attendance`-тэй хамааралтай. Шугаман регрессээр (`np.polyfit`) дутуу оноог таамагла.

1. `attendance_num` ба `score_clean` хоёулаа байгаа мөрүүдээр `np.polyfit(x, y, 1)` ашиглан налуу (slope), огтлол (intercept) ол.
2. `attendance_num` байгаа, `score_clean` дутуу мөрийн оноог таамагла.
3. Хоёр хувьсагчийн корреляцийг (`corr`) хэвлэ. Яагаад корреляц өндөр бол регресс нөхөлт сайн ажилладаг вэ? (үгээр)

**Хариу:** Корреляц 0.34 бага байна. Корреляц өндөр байвал ирц оноог сайн тайлбарлаж, таамаглал нарийн болно. Энд r² ≈ 0.12 тул регресс нөхөлт хязгаартай.

In [15]:
known = work.dropna(subset=["attendance_num", "score_clean"])
known = known[known["attendance_num"].between(0, 100)]
slope, intercept = np.polyfit(known["attendance_num"], known["score_clean"], 1)
print("slope:", round(slope, 3), "intercept:", round(intercept, 1))
print("corr:", round(known["attendance_num"].corr(known["score_clean"]), 2))
need = work["score_clean"].isna() & work["attendance_num"].between(0, 100)
work.loc[need, "score_reg"] = slope * work.loc[need, "attendance_num"] + intercept
print("таамагласан мөр:", need.sum())
work.loc[need, ["id", "attendance_num", "score_reg"]].head()

slope: 0.476 intercept: 32.0
corr: 0.34
таамагласан мөр: 34


     id  attendance_num  score_reg
2   103            86.0  72.960711
4   104            80.0  70.106035
11  111            91.0  75.339607
17  117            90.0  74.863828
23  123            75.0  67.727139

## 9) Data leakage-гүй цэвэрлэгээ

Хэрэв дундаж/медианыг **бүх** өгөгдлөөс тооцоод дараа нь train/test болговол test-ийн мэдээлэл train руу нэвтэрнэ. Үүнийг **data leakage** гэнэ.

Зөв дараалал:

1. Эхлээд train/test хуваана
2. Нөхөх утгыг **зөвхөн train**-ээс тооцно
3. Тэр утгыг train болон test хоёуланд хэрэглэнэ

### TASK 14: Train/test хуваалт

1. `work`-ийг 80% train, 20% test болго (`sample(frac=0.8, random_state=42)`).
2. Train-ийн `score_clean` медианыг тооц.
3. Test-ийн дутуу оноог **train-ийн** медианаар нөх.
4. Train-ийн медиан ба бүх өгөгдлийн медиан ялгаатай байж болох уу? Яагаад зөв арга нь train-ийнхийг ашигладаг вэ? (үгээр)

**Хариу:** Train ба бүх өгөгдлийн медиан ялгаатай байж болно (74.0 ба 73.5). Test-ийн мэдээллийг нөхөх утгад оруулахгүйн тулд зөвхөн train-ийн медианыг ашиглана.

In [16]:
train = work.sample(frac=0.8, random_state=42)
test = work.drop(train.index)
train_med = train["score_clean"].median()
test_filled = test["score_clean"].fillna(train_med)
print("train:", len(train), "test:", len(test))
print("train медиан:", train_med, "бүх өгөгдлийн медиан:", work["score_clean"].median())
print("test NaN (өмнө -> дараа):", test["score_clean"].isna().sum(), "->", test_filled.isna().sum())

train: 242 test: 61
train медиан: 74.0 бүх өгөгдлийн медиан: 73.5
test NaN (өмнө -> дараа): 8 -> 0


## 10) Бүх алхмыг нэг функцэд нэгтгэх

Цэвэрлэгээг давтагдах боломжтой болгохын тулд нэг функц болгоно.

### TASK 15: clean_students функц

Функц дараах алхмуудыг дарааллаар нь хийнэ:

1. `?`, `N/A`, хоосон зайг `NaN` болгох
2. `age`, `score`, `attendance`-ийг тоон болгох (`coerce`)
3. Мужаас гарсан утгыг `NaN` болгох (нас 16 ба 40, оноо болон ирц 0 ба 100)
4. `city`-г стандартчилах
5. Бүрэн давхардлыг арилгах

Эцэст нь цэвэрлэгээний өмнө ба дараах мөр, дутуу утгын тоог харьцуул.

In [17]:
def clean_students(path):
    d = pd.read_csv(path, na_values=["?", "N/A"])
    d = d.replace(r"^\s*$", pd.NA, regex=True)
    for c in ["age", "score", "attendance"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d["age"] = d["age"].where(d["age"].between(16, 40))
    d["score"] = d["score"].where(d["score"].between(0, 100))
    d["attendance"] = d["attendance"].where(d["attendance"].between(0, 100))
    d["city"] = d["city"].str.strip().str.lower()
    d["city"] = d["city"].replace({"уб": "улаанбаатар"})
    d = d.drop_duplicates()
    return d

clean = clean_students("students.csv")

print("мөр:", len(raw), "->", len(clean))
print("NaN өмнө:", df[["age_num", "score_num", "attendance_num"]].isna().sum().sum())
print(clean.isna().sum())

мөр: 315 -> 303
NaN өмнө: 74
id              0
age            41
score          37
attendance     14
city            9
program         0
enroll_date     4
dtype: int64


## 11) Дүгнэлт асуултууд (Concept Check)

### TASK 16: Verbal

1. `N/A`, `?`, `0` гурвын аль нь дутуу утга, аль нь жинхэнэ утга вэ? Яагаад?
2. Оноо `900` гэсэн утгыг устгах уу, засах уу, нөхөх үү? Шалтгаанаа бич.
3. Дизайн хөтөлбөр дээр дутуу оноо илүү олон байгаа нь шинжилгээнд ямар нөлөөтэй вэ?
4. Яагаад дутуу утгыг шууд нөхөхөөс **өмнө** шалтгааныг нь судлах хэрэгтэй вэ?

**Хариу:**
1. `N/A` ба `?` нь утга байхгүйг заадаг тэмдэг тул дутуу утга. `0` нь 0–100 мужийн бодит оноо тул жинхэнэ утга.
2. `900` оноог засахгүй, NaN болгож нөхсөн. Засах нь 90 гэсэн таамаглал; устгавал дутуу тоо өсөх тул нөхөх нь илүү найдвартай.
3. Дизайн хөтөлбөрт дутуу оноо илүү тул нийт медианаар нөхвөл Дизайны дундаж өсөж, хөтөлбөрийн бодит зөрүү алдагдана.
4. Дутуу байдлын шалтгааныг эхлээд судлах ёстой. MAR эсвэл MNAR бол энгийн нөхөлт дүгнэлтийг гажуулна.

### Тайлбар
- Бүх TODO гүйцэтгэж, код ажиллуулсан. Үр дүнг тайланд тусгасан.

# Data Mining - Lab 5 Тайлан
## Өгөгдлийн чанар: дутуу ба алдаатай өгөгдөл илрүүлэх

**Нэр:** А.Сүхболд  
**Оюутны код:** s22c055b  
**Хичээл:** Өгөгдлийн уурхай  
**Огноо:** 2026 он 10 сар 21 өдөр  

---

## 1. Lab зорилго
students.csv өгөгдлийн чанарыг шалгаж, дутуу ба буруу утга, давхардал, бичлэгийн зөрүүг илрүүлэн тохирох аргаар цэвэрлэх. Цэвэрлэгээг train/test хуваалтын дараа хийж, data leakage-ээс сэргийлэх.

---

## 2. Dataset-ийн товч тайлбар
- Dataset нэр: students.csv
- Мөр, баганын тоо: 315 мөр, 7 багана
- Ашигласан үндсэн баганууд: id, age, score, attendance, city, program, enroll_date

Оюутны нас, оноо, ирц, хот, хөтөлбөр, элссэн огноог агуулна. Дутуу утга, буруу төрлийн утга, давхардал, хотын нэрийн зөрүү, огнооны янз бүрийн формат зориудаар оруулсан.

---

## 3. Дутуу утга

### 3.1 Илрүүлэлт
- Хамгийн их дутуу утгатай багана: age (26, 8.3%) ба score (26, 8.3%). Буруу төрлийн утгыг нэмбэл age 32, score 29.
- Нууц дутуу утга: city баганад хоосон зайгаас бүрдсэн 5 утга илэрсэн. Дутуу утгын тоо 4-өөс 9 болсон.

### 3.2 Механизм
- Дутуу score нь MAR гэж үзсэн. Дизайн хөтөлбөрт 19.3%, Программчлалд 4.8%, Сүлжээнд 5.8% дутуу байгаа тул дутуу байдал хөтөлбөрөөс хамаарч байна. MNAR-ийг өгөгдлөөс бүрэн үгүйсгэх боломжгүй.

---

## 4. Алдаатай утга
- Буруу төрлийн утгууд: age-д «хорь» (4), «twenty» (2); score-д «67%», «87%», «59%» (3).
- Мужаас гарсан утгууд: age 12 (-5, -1, 45, 52, 60, 150, 190), score 10 (-10, 110, 150, 900), attendance 2 (105, 120).
- city баганын нийцгүй байдлыг strip() ба lower()-ээр арилгаж, «уб»-ыг «улаанбаатар» болгосон. 9 өөр бичлэг 4 хот болсон.
- Огнооны формат 3 байсан: YYYY/MM/DD, YYYY-MM-DD, DD/MM/YYYY. Жилийн орон буруу 2 утга (20250/09/25, 20250/09/12) амжилтгүй болсон.

---

## 5. Давхардал ба хэтрэлт
- Бүрэн давхардсан мөр: 12 (315 → 303 мөр).
- Зөрчилтэй id: 3 (183, 197, 268).
- IQR аргаар олсон хэтрэлт: age 10 мөр (хязгаар 16–24); score 15 мөр (хязгаар 51.5–95.5).
- z-score (|z| > 2.5) аргаар олсон хэтрэлт: score 5 мөр (0, 0, 0, 7, 12).

---

## 6. Засварын шийдвэр

| Асуудал | Сонгосон арга | Шалтгаан |
|---|---|---|
| Дутуу score | Хөтөлбөрийн медианаар нөхсөн (train-ээс тооцсон) | Хөтөлбөр бүрийн оноо ялгаатай; нийт медиан Дизайн дундажийг сулруулна |
| Мужаас гарсан score | NaN болгож нөхсөн | 900 гэх мэт утгыг засахгүй; засах нь таамаглал болно |
| Давхардал | Бүрэн давхардлыг устгасан; зөрчилтэй id-г тэмдэглэсэн | Ижил мөр давхардсан; зөрчилтэй id-ийн аль нь зөв болохыг тогтоох боломжгүй |
| city | strip(), lower(), «уб» → «улаанбаатар» | Ижил хотыг нэг утгад нэгтгэх |

---

## 7. Reflection
1. Цэвэрлэгээг train/test хуваасны дараа хийх ёстой. Нөхөх статистикийг зөвхөн train-ээс авснаар test-ийн мэдээлэл сургалтад орохгүй.
2. clean_students() функцэд алхмуудыг нэгтгэсэн нь дараагийн lab-д өгөгдлийг адил дарааллаар бэлтгэх суурь болно.
3. Хэтрэлт ба мужийн алдааг ялгах асуудал харьцангуй төвөгтэй байсан. Статистикийн хэтрэлт бүр алдаа биш тул хоёр шалгуурыг хамт авч үзэх шаардлагатай.

---

## 8. Дүгнэлт
Өгөгдлийн цэвэрлэгээнд зөвхөн isna() хангалтгүй; хоосон зай, текст тоо, бичилтийн зөрүүг давхар шалгах хэрэгтэй. Дутуу score-ийг хөтөлбөрийн медианаар нөхөх нь нийт медианаас илүү бүлгийн бодит дундажийг хадгална. Утга бүрийг шууд устгахаас эсвэл засахаас өмнө алдаа мөн эсэхийг шалгах нь үр дүнг илүү найдвартай болгоно.
